In [0]:
-- ============================================================
-- AEGIS — GOLD LAYER
-- ============================================================
-- Purpose:
-- Business-facing analytical layer for:
--   1. Supplier performance
--   2. Delivery performance
--   3. Inventory health
--   4. Warehouse performance
--   5. Carrier performance
--   6. Supply-chain risk
--
-- Design principles:
--   - Deterministic SQL only
--   - No GenAI
--   - No ground-truth incident leakage
--   - Preserve time trends
--   - Metrics are derived exclusively from Silver
-- ============================================================


-- ============================================================
-- 1. SUPPLIER PERFORMANCE
-- Grain: supplier + month
-- ============================================================

CREATE OR REPLACE TABLE workspace.aegis_gold.supplier_performance
USING DELTA
AS
WITH monthly_orders AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        oi.supplier_id,
        COUNT(DISTINCT o.order_id) AS orders,
        SUM(oi.quantity) AS units_supplied,
        SUM(oi.revenue) AS revenue
    FROM workspace.aegis_silver.order o
    JOIN workspace.aegis_silver.order_item oi
        ON o.order_id = oi.order_id
    GROUP BY
        DATE_TRUNC('month', o.order_date),
        oi.supplier_id
),

supplier_products AS (
    SELECT
        supplier_id,
        COUNT(DISTINCT product_id) AS product_count,
        AVG(agreed_lead_time) AS avg_agreed_lead_time
    FROM workspace.aegis_silver.supplier_product
    GROUP BY supplier_id
),

supplier_delivery AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS month,
        oi.supplier_id,
        COUNT(DISTINCT s.shipment_id) AS shipments,
        AVG(s.actual_transit_days) AS avg_actual_transit_days,
        AVG(s.delivery_delay_days) AS avg_delivery_delay_days,
        AVG(CASE WHEN s.is_late THEN 1.0 ELSE 0.0 END) AS late_delivery_rate
    FROM workspace.aegis_silver.order o
    JOIN workspace.aegis_silver.order_item oi
        ON o.order_id = oi.order_id
    JOIN workspace.aegis_silver.shipment_item si
        ON oi.order_item_id = si.order_item_id
    JOIN workspace.aegis_silver.shipment s
        ON si.shipment_id = s.shipment_id
    GROUP BY
        DATE_TRUNC('month', o.order_date),
        oi.supplier_id
)

SELECT
    mo.month,
    mo.supplier_id,
    sp.supplier_name,
    sp.region,
    sp.supplier_type,
    sp.reliability_score,
    sp.lead_time_variability,

    mo.orders,
    mo.units_supplied,
    mo.revenue,

    COALESCE(sd.shipments, 0) AS shipments,
    COALESCE(sd.avg_actual_transit_days, 0) AS avg_actual_transit_days,
    COALESCE(sd.avg_delivery_delay_days, 0) AS avg_delivery_delay_days,
    COALESCE(sd.late_delivery_rate, 0) AS late_delivery_rate,

    spp.product_count,
    spp.avg_agreed_lead_time

FROM monthly_orders mo

JOIN workspace.aegis_silver.supplier sp
    ON mo.supplier_id = sp.supplier_id

LEFT JOIN supplier_delivery sd
    ON mo.month = sd.month
    AND mo.supplier_id = sd.supplier_id

LEFT JOIN supplier_products spp
    ON mo.supplier_id = spp.supplier_id;


-- ============================================================
-- 2. DELIVERY PERFORMANCE
-- Grain: month + warehouse + carrier
-- ============================================================

CREATE OR REPLACE TABLE workspace.aegis_gold.delivery_performance
USING DELTA
AS
SELECT
    DATE_TRUNC('month', s.shipment_date) AS month,

    s.warehouse_id,
    w.region AS warehouse_region,
    s.carrier_id,
    c.carrier_name,
    c.service_type,

    COUNT(*) AS shipment_count,

    SUM(
        CASE
            WHEN s.is_late THEN 1
            ELSE 0
        END
    ) AS late_shipments,

    ROUND(
        AVG(
            CASE
                WHEN s.is_late THEN 1.0
                ELSE 0.0
            END
        ),
        4
    ) AS late_delivery_rate,

    ROUND(
        AVG(s.actual_transit_days),
        2
    ) AS avg_actual_transit_days,

    ROUND(
        AVG(s.expected_transit_days),
        2
    ) AS avg_expected_transit_days,

    ROUND(
        AVG(s.delivery_delay_days),
        2
    ) AS avg_delivery_delay_days,

    ROUND(
        MAX(s.delivery_delay_days),
        2
    ) AS max_delivery_delay_days

FROM workspace.aegis_silver.shipment s

JOIN workspace.aegis_silver.warehouse w
    ON s.warehouse_id = w.warehouse_id

JOIN workspace.aegis_silver.carrier c
    ON s.carrier_id = c.carrier_id

GROUP BY
    DATE_TRUNC('month', s.shipment_date),
    s.warehouse_id,
    w.region,
    s.carrier_id,
    c.carrier_name,
    c.service_type;


-- ============================================================
-- 3. INVENTORY HEALTH
-- Grain: date + warehouse + product
-- ============================================================

CREATE OR REPLACE TABLE workspace.aegis_gold.inventory_health
USING DELTA
AS
SELECT
    i.snapshot_date,
    i.warehouse_id,
    i.product_id,

    p.product_name,
    p.category,

    w.region AS warehouse_region,
    w.capacity AS warehouse_capacity,

    i.stock_quantity,
    i.reserved_quantity,
    i.available_quantity,
    i.stockout_flag,

    CASE
        WHEN i.stockout_flag THEN 1
        ELSE 0
    END AS stockout_indicator,

    p.demand_index,

    ROUND(
        CASE
            WHEN p.demand_index > 0
            THEN i.available_quantity / p.demand_index
            ELSE NULL
        END,
        2
    ) AS inventory_demand_ratio

FROM workspace.aegis_silver.inventory i

JOIN workspace.aegis_silver.product p
    ON i.product_id = p.product_id

JOIN workspace.aegis_silver.warehouse w
    ON i.warehouse_id = w.warehouse_id;


-- ============================================================
-- 4. WAREHOUSE PERFORMANCE
-- Grain: warehouse + month
-- ============================================================

CREATE OR REPLACE TABLE workspace.aegis_gold.warehouse_performance
USING DELTA
AS
WITH shipments AS (
    SELECT
        DATE_TRUNC('month', shipment_date) AS month,
        warehouse_id,

        COUNT(*) AS shipments,

        SUM(
            CASE
                WHEN is_late THEN 1
                ELSE 0
            END
        ) AS late_shipments,

        AVG(
            CASE
                WHEN is_late THEN 1.0
                ELSE 0.0
            END
        ) AS late_delivery_rate,

        AVG(actual_transit_days) AS avg_actual_transit_days,
        AVG(delivery_delay_days) AS avg_delivery_delay_days

    FROM workspace.aegis_silver.shipment

    GROUP BY
        DATE_TRUNC('month', shipment_date),
        warehouse_id
),

inventory AS (
    SELECT
        DATE_TRUNC('month', snapshot_date) AS month,
        warehouse_id,

        AVG(stock_quantity) AS avg_stock_quantity,
        AVG(available_quantity) AS avg_available_quantity,

        AVG(
            CASE
                WHEN stockout_flag THEN 1.0
                ELSE 0.0
            END
        ) AS stockout_rate

    FROM workspace.aegis_silver.inventory

    GROUP BY
        DATE_TRUNC('month', snapshot_date),
        warehouse_id
)

SELECT
    s.month,
    s.warehouse_id,

    w.region,
    w.warehouse_type,
    w.capacity,
    w.processing_efficiency,

    s.shipments,
    s.late_shipments,
    ROUND(s.late_delivery_rate, 4) AS late_delivery_rate,
    ROUND(s.avg_actual_transit_days, 2) AS avg_actual_transit_days,
    ROUND(s.avg_delivery_delay_days, 2) AS avg_delivery_delay_days,

    ROUND(i.avg_stock_quantity, 2) AS avg_stock_quantity,
    ROUND(i.avg_available_quantity, 2) AS avg_available_quantity,
    ROUND(i.stockout_rate, 4) AS stockout_rate

FROM shipments s

JOIN workspace.aegis_silver.warehouse w
    ON s.warehouse_id = w.warehouse_id

LEFT JOIN inventory i
    ON s.month = i.month
    AND s.warehouse_id = i.warehouse_id;


-- ============================================================
-- 5. CARRIER PERFORMANCE
-- Grain: carrier + month
-- ============================================================

CREATE OR REPLACE TABLE workspace.aegis_gold.carrier_performance
USING DELTA
AS
SELECT
    DATE_TRUNC('month', s.shipment_date) AS month,

    s.carrier_id,
    c.carrier_name,
    c.service_type,

    c.base_transit_days,
    c.transit_variability,

    COUNT(*) AS shipment_count,

    SUM(
        CASE
            WHEN s.is_late THEN 1
            ELSE 0
        END
    ) AS late_shipments,

    ROUND(
        AVG(
            CASE
                WHEN s.is_late THEN 1.0
                ELSE 0.0
            END
        ),
        4
    ) AS late_delivery_rate,

    ROUND(
        AVG(s.actual_transit_days),
        2
    ) AS avg_actual_transit_days,

    ROUND(
        AVG(s.delivery_delay_days),
        2
    ) AS avg_delivery_delay_days,

    ROUND(
        STDDEV(s.actual_transit_days),
        2
    ) AS transit_time_stddev

FROM workspace.aegis_silver.shipment s

JOIN workspace.aegis_silver.carrier c
    ON s.carrier_id = c.carrier_id

GROUP BY
    DATE_TRUNC('month', s.shipment_date),
    s.carrier_id,
    c.carrier_name,
    c.service_type,
    c.base_transit_days,
    c.transit_variability;


-- ============================================================
-- 6. SUPPLY CHAIN RISK
-- Grain: month + supplier
--
-- IMPORTANT:
-- This is a deterministic SIGNAL table.
-- It does NOT use ground-truth incidents.
-- ============================================================

CREATE OR REPLACE TABLE workspace.aegis_gold.supply_chain_risk
USING DELTA
AS
WITH supplier_metrics AS (

    SELECT
        month,
        supplier_id,

        late_delivery_rate,
        avg_delivery_delay_days,
        avg_actual_transit_days,
        avg_agreed_lead_time,

        reliability_score,
        lead_time_variability

    FROM workspace.aegis_gold.supplier_performance
),

inventory_metrics AS (

    SELECT
        DATE_TRUNC('month', i.snapshot_date) AS month,

        sp.supplier_id,

        AVG(
            CASE
                WHEN i.stockout_flag THEN 1.0
                ELSE 0.0
            END
        ) AS stockout_rate

    FROM workspace.aegis_silver.inventory i

    JOIN workspace.aegis_silver.supplier_product sp
        ON i.product_id = sp.product_id

    GROUP BY
        DATE_TRUNC('month', i.snapshot_date),
        sp.supplier_id
)

SELECT
    sm.month,
    sm.supplier_id,

    sm.reliability_score,
    sm.lead_time_variability,

    ROUND(sm.late_delivery_rate, 4)
        AS late_delivery_rate,

    ROUND(sm.avg_delivery_delay_days, 2)
        AS avg_delivery_delay_days,

    ROUND(sm.avg_actual_transit_days, 2)
        AS avg_actual_transit_days,

    ROUND(sm.avg_agreed_lead_time, 2)
        AS avg_agreed_lead_time,

    ROUND(
        COALESCE(im.stockout_rate, 0),
        4
    ) AS stockout_rate,

    /*
     * Signal components.
     * These are intentionally transparent rather than
     * being a black-box ML score.
     */

    CASE
        WHEN sm.late_delivery_rate >= 0.30 THEN 2
        WHEN sm.late_delivery_rate >= 0.15 THEN 1
        ELSE 0
    END AS delivery_risk_signal,

    CASE
        WHEN sm.avg_delivery_delay_days >= 3 THEN 2
        WHEN sm.avg_delivery_delay_days >= 1 THEN 1
        ELSE 0
    END AS delay_risk_signal,

    CASE
        WHEN COALESCE(im.stockout_rate, 0) >= 0.10 THEN 2
        WHEN COALESCE(im.stockout_rate, 0) >= 0.05 THEN 1
        ELSE 0
    END AS inventory_risk_signal,

    CASE
        WHEN sm.reliability_score < 0.70 THEN 2
        WHEN sm.reliability_score < 0.85 THEN 1
        ELSE 0
    END AS supplier_reliability_signal,

    (
        CASE
            WHEN sm.late_delivery_rate >= 0.30 THEN 2
            WHEN sm.late_delivery_rate >= 0.15 THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN sm.avg_delivery_delay_days >= 3 THEN 2
            WHEN sm.avg_delivery_delay_days >= 1 THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN COALESCE(im.stockout_rate, 0) >= 0.10 THEN 2
            WHEN COALESCE(im.stockout_rate, 0) >= 0.05 THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN sm.reliability_score < 0.70 THEN 2
            WHEN sm.reliability_score < 0.85 THEN 1
            ELSE 0
        END
    ) AS risk_signal_score

FROM supplier_metrics sm

LEFT JOIN inventory_metrics im
    ON sm.month = im.month
    AND sm.supplier_id = im.supplier_id;